# Neural networks, demystified: a live demo

**Plan**
1. **A neuron is tiny**: a weighted sum and a squiggle.
2. **Why the squiggle matters**: without it, any number of layers collapses into one.
3. **Demo 1: untangling spirals**: watch a network bend space.
4. **Demo 2: an image is a function**: teach a network `f(x, y) → (r, g, b)`.
   - spectral bias: why plain networks stay blurry
   - generalization vs. memorization: hide half the pixels
   - "infinite" resolution

> The ML code (models, training loops) is in this notebook. The widgets and plotting live in `nn_demo_utils.py`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from IPython.display import display

from nn_demo_utils import (spiral_dashboard, ImageFitDashboard, compare_grid,
                           load_image, make_coords, arr_to_png, masked_view)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True  # faster matmuls on RTX 30xx+, plenty precise here

# Size profile: the image demos are ~16x cheaper on CPU with these settings (about 30 s per run).
# The sweet-spot Fourier frequency scales with image resolution, hence the smaller sigma on CPU.
ON_CPU = device == "cpu"
SIDE, WIDTH, STEPS, SIGMA = (96, 128, 600, 4) if ON_CPU else (256, 256, 1500, 10)   # px / neurons per layer / steps / sigma
print("running on:", "CPU (small, fast profile)" if ON_CPU else torch.cuda.get_device_name(0))

## 1 · A neuron is tiny

A neuron takes some numbers, multiplies each by a **weight**, adds them up with a **bias**, and passes the result through a nonlinear "squiggle" (here ReLU = `max(0, x)`).

The weights and bias are the only things that ever get learned.

In [ ]:
neuron = nn.Linear(3, 1)                      # 3 inputs -> 1 output:  w·x + b
x = torch.tensor([[0.5, -1.0, 2.0]])

print("weights :", neuron.weight.data.numpy().round(3))
print("bias    :", neuron.bias.data.numpy().round(3))
print("w·x + b        =", round(neuron(x).item(), 4))
print("ReLU(w·x + b)  =", round(torch.relu(neuron(x)).item(), 4))

## 2 · Why the squiggle matters

Stack three linear layers **without** an activation. The result is still just *one* linear function: the matrices multiply together.

In [ ]:
f = nn.Sequential(nn.Linear(2, 64), nn.Linear(64, 64), nn.Linear(64, 2))   # no activations!

W_total = f[2].weight @ f[1].weight @ f[0].weight
b_total = f[2].weight @ (f[1].weight @ f[0].bias + f[1].bias) + f[2].bias

x = torch.randn(5, 2)
print("3 layers == one 2x2 matrix + bias ?", torch.allclose(f(x), x @ W_total.T + b_total, atol=1e-5))
print(f"{sum(p.numel() for p in f.parameters()):,} parameters... that can only draw a straight line")

## 3 · Demo 1: untangling spirals

Two interleaved spirals, blue and orange. The network gets a point `(x, y)` and must say *which spiral it belongs to*. A straight line can never separate them.

First, the building block. An **MLP** (multi-layer perceptron) is just layers of neurons stacked on top of each other:

In [ ]:
ACTIVATIONS = {"None (linear)": None, "ReLU": nn.ReLU, "Tanh": nn.Tanh}

def make_mlp(in_dim, out_dim, width, depth, activation=nn.ReLU):
    layers, d = [], in_dim
    for _ in range(depth):
        layers.append(nn.Linear(d, width))
        if activation is not None:
            layers.append(activation())
        d = width
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers)

make_mlp(2, 2, width=32, depth=3)

### The training loop: the whole of "learning" in 4 lines

In [ ]:
def train_spirals(model, X, y, steps, lr, on_frame=None, frame_every=25):
    X, y = torch.from_numpy(X).to(device), torch.from_numpy(y).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for step in range(steps + 1):
        logits = model(X)                        # 1. forward:  make a guess
        loss = F.cross_entropy(logits, y)        # 2. loss:     how wrong is it?
        opt.zero_grad(); loss.backward()         # 3. backward: who is to blame? (autodiff)
        opt.step()                               # 4. update:   nudge every weight a little

        if on_frame and step % frame_every == 0:
            acc = (logits.argmax(1) == y).float().mean().item()
            on_frame(step, loss.item(), acc)

**Try:**
- `None (linear)`: it can only draw a straight line, however many layers you add.
- `ReLU`: the boundary is built from straight pieces (folds).
- `Tanh`: smooth curves.
- 1 hidden layer with 4 neurons: not enough capacity.

In [ ]:
display(spiral_dashboard(make_mlp, train_spirals, ACTIVATIONS, device))

## 4 · Demo 2: an image is a function

Treat a picture as a function **`f(x, y) → (r, g, b)`**:

| | |
|---|---|
| **input** | the pixel's coordinates `(x, y)`, scaled to `[0, 1]` |
| **output** | its colour `(r, g, b)`, each in `[0, 1]` |
| **dataset** | one image: every pixel is one training example |
| **loss** | mean squared error between predicted and true colour |

The network has to *memorize* the image. That's the goal here, and it lets us **see the entire learned function at once**.

In [ ]:
img = load_image("cats_1.png", max_side=SIDE)
h, w, _ = img.shape
coords = make_coords(h, w, device)

print("image:", img.shape, "->", h * w, "training examples")
print("first 3 inputs  (x, y)   :", np.round(coords[:3].cpu().numpy().astype(float), 4).tolist())
print("first 3 targets (r, g, b):", np.round(img.reshape(-1, 3)[:3].astype(float), 3).tolist())

### Two ways to feed in the coordinates

- **raw**: give the network `(x, y)` as is.
- **Fourier features**: first map `(x, y)` to many sines and cosines at random frequencies. This gives the network ready-made "wiggly" building blocks.

In [ ]:
class RawCoords(nn.Module):
    out_dim = 2
    def forward(self, v):
        return v

class FourierFeatures(nn.Module):
    # (x, y) -> [sin(2π·Bv), cos(2π·Bv)],  random frequencies B ~ N(0, σ²)
    def __init__(self, n_freqs=256, sigma=10.0):
        super().__init__()
        self.register_buffer("B", torch.randn(2, n_freqs) * sigma)
        self.out_dim = 2 * n_freqs
    def forward(self, v):
        proj = 2 * torch.pi * v @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

### Training: the same 4-line loop as before

In [ ]:
encode = FourierFeatures(sigma=10.0).to(device)
encode.B.shape

In [ ]:
def fit_image(img, train_mask=None, encoding="raw", sigma=SIGMA, width=WIDTH, depth=4,
              steps=STEPS, lr=1e-3, on_frame=None, frame_every=25, seed=0):
    torch.manual_seed(seed)
    h, w, _ = img.shape
    coords = make_coords(h, w, device)                          # inputs : (h*w, 2)
    colors = torch.from_numpy(img.reshape(-1, 3)).to(device)    # targets: (h*w, 3)
    if train_mask is None:
        train_mask = np.ones(h * w, bool)
    mask = torch.from_numpy(train_mask).to(device)              # which pixels we may learn from

    encode = FourierFeatures(sigma=sigma).to(device) if encoding == "fourier" else RawCoords()
    model = nn.Sequential(make_mlp(encode.out_dim, 3, width, depth, nn.ReLU), nn.Sigmoid()).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)   # take smaller steps near the end

    for step in range(steps + 1):
        pred = model(encode(coords[mask]))                      # 1. forward
        loss = F.mse_loss(pred, colors[mask])                   # 2. loss
        opt.zero_grad(); loss.backward()                        # 3. backward
        opt.step(); sched.step()                                # 4. update

        if on_frame and (step % frame_every == 0 or step == steps):
            with torch.no_grad():                               # ask for EVERY pixel's colour
                full = model(encode(coords))
            test_mse = F.mse_loss(full[~mask], colors[~mask]).item() if (~mask).any() else None
            on_frame(step, full.reshape(h, w, 3).cpu().numpy(), loss.item(), test_mse, n_params)
    return model, encode

### Playground

Start with **raw (x, y)** on the cat photo and press **Train**. Then switch to **Fourier features**. Then try the line-art images.

In [ ]:
dash = ImageFitDashboard(fit_image, device, default_image="cats_1.png",
                         res=SIDE, width=WIDTH, steps=STEPS, sigma=SIGMA)
display(dash.widget())

## 5 · Experiment: spectral bias

Same network, same training. The only difference is **how the input is represented**.

Neural networks learn smooth, low-frequency patterns first, and struggle with sharp detail. Thin lines are the worst case.

In [ ]:
line_cat = load_image("cat-line-art.webp", max_side=SIDE)
_ = compare_grid(fit_image, line_cat, [
    ("raw (x, y)",          dict(encoding="raw")),
    ("Fourier features",    dict(encoding="fourier", sigma=SIGMA)),
], device, steps=STEPS)

## 6 · Experiment: memorization vs. generalization

We **hide a random 50% of the pixels** during training (the checkerboard). Afterwards, we ask the network for the colours of the hidden pixels: our test set.

- low σ: smooth guesses, a bit blurry
- medium σ: good guesses for pixels it never saw
- high σ: **perfect on training pixels, noise on hidden ones**. That's overfitting, made visible.

In [ ]:
photo = load_image("cats_2.png", max_side=SIDE)
mask = np.random.default_rng(0).uniform(size=photo.shape[0] * photo.shape[1]) < 0.5

_ = compare_grid(fit_image, photo, [
    (f"σ = {s}", dict(encoding="fourier", sigma=s)) for s in (1, SIGMA, 100)
], device, mask=mask, steps=STEPS, thumb=260)

## 7 · Bonus: "infinite" resolution

The network isn't a pixel grid. It's a continuous function, so we can ask for colours *between* the original pixels and render at 4× the training resolution.

In [ ]:
import ipywidgets as W

def render(model, encode, h, w):
    with torch.no_grad():
        c = make_coords(h, w, device)
        out = torch.cat([model(encode(chunk)) for chunk in c.split(65536)])
    return out.reshape(h, w, 3).cpu().numpy()

mountain = load_image("Line-mountain-723x1024.jpeg", max_side=96 if ON_CPU else 192)
model, encode = fit_image(mountain, encoding="fourier", sigma=SIGMA, steps=STEPS)

h, w, _ = mountain.shape
k = 4
big = render(model, encode, h * k, w * k)
pixelated = np.repeat(np.repeat(mountain, k, 0), k, 1)

y0, x0, s = h * k // 3, w * k // 3, 160      # crop the same region from both
display(W.HBox([
    W.VBox([W.HTML("<b>training pixels, zoomed 4×</b>"),
            W.Image(value=arr_to_png(pixelated[y0:y0+s, x0:x0+s]), width=380)]),
    W.VBox([W.HTML("<b>network rendered at 4×</b>"),
            W.Image(value=arr_to_png(big[y0:y0+s, x0:x0+s]), width=380)]),
]))

## Takeaways

1. A neural network is **a program whose logic is found by search** (gradient descent), not written by hand.
2. The **nonlinearity** is what lets it bend space. Without it, depth is useless.
3. Training is always the same loop: **forward → loss → backward → update**.
4. **Representation matters**: the same network went from blurry to sharp just by changing the input encoding.
5. **Low training loss ≠ a good model**. Always measure on data the model has never seen.

The same idea in 3D, `(x, y, z, view direction) → (colour, density)`, is **NeRF**: 3D scenes reconstructed from photos.